In [14]:
#Imports 

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

In [15]:
# Setup and data loading 

df = pd.read_csv('../data/processed/wine_combined.csv')

params = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
          'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
          'density', 'pH', 'sulphates', 'alcohol']

# OIV / EU Regulation 606/2009 spec limits
spec_limits = {
    'volatile acidity':    {'LSL': 0.08,  'USL': 1.2},   # OIV / US Federal
    'pH':                  {'LSL': 2.9,   'USL': 4.0},   # UC Davis
    'sulphates':           {'LSL': 0.25,  'USL': 1.5},   # Practical range
    'alcohol':             {'LSL': 8.5,   'USL': 15.0},  # EU Reg 1308/2013
    'free sulfur dioxide': {'LSL': 10.0,  'USL': 60.0},  # OIV Annex C
    'chlorides':           {'LSL': 0.005, 'USL': 0.20},  # Practical range
}

WINE_COLORS = {'Red': 'crimson', 'White': 'palegoldenrod'}

print(f"Dataset loaded: {df.shape}")
print(f"Spec limits defined for {len(spec_limits)} parameters")
print("✅ Ready")

Dataset loaded: (6497, 16)
Spec limits defined for 6 parameters
✅ Ready


In [16]:
# #I MR Control Charts

# def imr_chart(data, param, wine_type, spec):
#     subset = data[data['wine_type'] == wine_type][param].reset_index(drop=True)
    
#     mean = subset.mean()
#     std = subset.std()
#     ucl = mean + 3*std
#     lcl = mean - 3*std
    
#     # Flag OOC points
#     ooc = (subset > ucl) | (subset < lcl)
#     ooc_count = ooc.sum()
#     ooc_pct = round(ooc_count/len(subset)*100, 1)
    
#     fig = go.Figure()
    
#     # In control points
#     fig.add_trace(go.Scatter(
#         x=subset[~ooc].index,
#         y=subset[~ooc].values,
#         mode='markers',
#         name='In Control',
#         marker=dict(color='steelblue', size=4, opacity=0.5)
#     ))
    
#     # OOC points
#     fig.add_trace(go.Scatter(
#         x=subset[ooc].index,
#         y=subset[ooc].values,
#         mode='markers',
#         name='Out of Control',
#         marker=dict(color='red', size=6, symbol='x')
#     ))
    
#     # Control limit lines
#     fig.add_hline(y=mean, line_dash='solid', line_color='green', 
#                   annotation_text=f'Mean: {mean:.3f}')
#     fig.add_hline(y=ucl, line_dash='dash', line_color='red',
#                   annotation_text=f'UCL: {ucl:.3f}')
#     fig.add_hline(y=lcl, line_dash='dash', line_color='red',
#                   annotation_text=f'LCL: {lcl:.3f}')
    
#     # Spec limit lines
#     fig.add_hline(y=spec['USL'], line_dash='dot', line_color='orange',
#                   annotation_text=f'USL: {spec["USL"]}')
#     fig.add_hline(y=spec['LSL'], line_dash='dot', line_color='orange',
#                   annotation_text=f'LSL: {spec["LSL"]}')
    
#     fig.update_layout(
#         title=f'Control Chart — {param.title()} ({wine_type} Wine) | OOC: {ooc_count} points ({ooc_pct}%)',
#         xaxis_title='Sample Index',
#         yaxis_title=param.title(),
#         paper_bgcolor='#E8E8E8',
#         plot_bgcolor='white',
#         height=400,
#         legend=dict(orientation='h', y=-0.2)
#     )
    
#     fig.show()
#     return ooc_count, ooc_pct

# # Run for all parameters and both wine types
# results = []
# for param, spec in spec_limits.items():
#     for wtype in ['Red', 'White']:
#         ooc_count, ooc_pct = imr_chart(df, param, wtype, spec)
#         results.append({
#             'Parameter': param,
#             'Wine Type': wtype,
#             'OOC Count': ooc_count,
#             'OOC %': ooc_pct
#         })

IMR CHARTS SECTION - THIS PROVIDES INSIGHT INTO OOC...

In [17]:
# IMR Control Charts
def imr_chart(data, param, wine_type, spec):
    subset = data[data['wine_type'] == wine_type][param].reset_index(drop=True)
  
    mean = subset.mean()
    std = subset.std()
    ucl = mean + 3*std
    lcl = mean - 3*std
    
    # Flag OOC points
    ooc = (subset > ucl) | (subset < lcl)
    ooc_count = ooc.sum()
    ooc_pct = round(ooc_count/len(subset)*100, 1)
    
    fig = go.Figure()
    
    # In control points
    fig.add_trace(go.Scatter(
        x=subset[~ooc].index,
        y=subset[~ooc].values,
        mode='markers',
        name='In Control',
        marker=dict(color='steelblue', size=4, opacity=0.5)
    ))
    
    # OOC points
    fig.add_trace(go.Scatter(
        x=subset[ooc].index,
        y=subset[ooc].values,
        mode='markers',
        name='Out of Control',
        marker=dict(color='red', size=6, symbol='x')
    ))

    #Adding Control and Spec Limit Lines
    for y, dash, color, label in [
    (mean,         'solid', 'green',  f'Mean: {mean:.3f}'),
    (ucl,          'dash',  'red',    f'UCL: {ucl:.3f}'),
    (lcl,          'dash',  'red',    f'LCL: {lcl:.3f}'),
    (spec['USL'],  'dot',   'orange', f'USL: {spec["USL"]}'),
    (spec['LSL'],  'dot',   'orange', f'LSL: {spec["LSL"]}'),
    ]:
        fig.add_hline(y=y, line_dash=dash, line_color=color, annotation_text=label)

    
    # Three zone system — volatile acidity only
    if param == 'volatile acidity':
        fig.add_hline(y=0.7, line_dash='dashdot', line_color='purple',
                    annotation_text='⚠️ Sensory threshold: 0.7 g/L',
                    annotation_position='top right')
    
    #Shade the three zones
    for y0, y1, color, label in [
        (spec['LSL'], 0.7,            'green',  'Ideal zone'),
        (0.7,         spec['USL'],    'orange', 'Warning zone'),
        (spec['USL'], subset.max()+0.1,'red',   'Out of spec zone'),
    ]:
        fig.add_hrect(y0=y0, y1=y1, fillcolor=color, opacity=0.05,
                        annotation_text=label, annotation_position='right') 
        
 
    fig.update_layout(
        title=f'Control Chart — {param.title()} ({wine_type} Wine) | OOC: {ooc_count} points ({ooc_pct}%)',
        xaxis_title='Sample Index',
        yaxis_title=param.title(),
        paper_bgcolor='#E8E8E8',
        plot_bgcolor='white',
        height=400,
        legend=dict(orientation='h', y=-0.2)
    )
    
    fig.show()
    return ooc_count, ooc_pct



In [18]:
# Run for all parameters and both wine types
results = []
for param, spec in spec_limits.items():
    for wtype in ['Red', 'White']:
        ooc_count, ooc_pct = imr_chart(df, param, wtype, spec)
        results.append({
            'Parameter': param,
            'Wine Type': wtype,
            'OOC Count': ooc_count,
            'OOC %': ooc_pct
        })

results_df = pd.DataFrame(results)

print("OOC Summary — Are Our Processes Stable?")
print("=" * 55)
print(results_df.to_string(index=False))
print()

# Flag parameters with high OOC rates
print("\n⚠️ Parameters with OOC% above 5%:")
flagged = results_df[results_df['OOC %'] > 5]
if len(flagged) > 0:
    print(flagged.to_string(index=False))
else:
    print("None — all parameters within acceptable range")

OOC Summary — Are Our Processes Stable?
          Parameter Wine Type  OOC Count  OOC %
   volatile acidity       Red         10    0.6
   volatile acidity     White         81    1.7
                 pH       Red          8    0.5
                 pH     White         32    0.7
          sulphates       Red         27    1.7
          sulphates     White         48    1.0
            alcohol       Red          8    0.5
            alcohol     White          0    0.0
free sulfur dioxide       Red         22    1.4
free sulfur dioxide     White         32    0.7
          chlorides       Red         31    1.9
          chlorides     White        102    2.1


⚠️ Parameters with OOC% above 5%:
None — all parameters within acceptable range


CPK CALCULATION AND BAR CHARTS

In [19]:
# CPK CALCULATION — OIV/EU Reg 606/2009 Standards
# Using average moving range method (ASTM/ISC standard for individual measurements)
# d2 constant = 1.128 for subgroup size n=2

spec_limits = {
    'volatile acidity':    {'LSL': 0.08, 'USL': 1.2},   # OIV/US Federal limit
    'pH':                  {'LSL': 2.9,  'USL': 4.0},   # UC Davis / WineMakerMag
    'sulphates':           {'LSL': 0.25, 'USL': 1.5},   # Practical quality range
    'alcohol':             {'LSL': 8.5,  'USL': 15.0},  # EU Reg 1308/2013
    'free sulfur dioxide': {'LSL': 10.0, 'USL': 60.0},  # OIV Annex C
    'chlorides':           {'LSL': 0.005,'USL': 0.20},  # Practical quality range
}

d2 = 1.128  # control chart constant for subgroup size n=2

def calculate_cpk_proper(data, param, spec):
    values = data[param].dropna().reset_index(drop=True)
    
    mean = values.mean()
    
    # Step 1 — Check normality via skewness
    skewness = values.skew()
    normal_enough = abs(skewness) <= 1.0
    
    # Step 2 — Estimate sigma via average moving range (SPC standard)
    moving_range = values.diff().abs().dropna()
    avg_moving_range = moving_range.mean()
    sigma = avg_moving_range / d2
    
    # Step 3 — Calculate Cpu, Cpl, Cpk
    cpu = (spec['USL'] - mean) / (3 * sigma)
    cpl = (mean - spec['LSL']) / (3 * sigma)
    cpk = min(cpu, cpl)
    
    # Step 4 — Assign status using food industry thresholds
    if cpk >= 1.0:
        status = '✅ Capable'
    elif cpk >= 0.67:
        status = '⚠️ Marginal'
    else:
        status = '❌ Incapable'
    
    return {
        'Mean': round(mean, 3),
        'Sigma': round(sigma, 3),
        'Skewness': round(skewness, 3),
        'Normal Enough': '✅' if normal_enough else '⚠️ Skewed',
        'CPU': round(cpu, 3),
        'CPL': round(cpl, 3),
        'Cpk': round(cpk, 3),
        'Status': status
    }

# Run for all parameters and wine types
cpk_results = []

for param, spec in spec_limits.items():
    for wtype in ['Red', 'White']:
        subset = df[df['wine_type'] == wtype]
        result = calculate_cpk_proper(subset, param, spec)
        result['Parameter'] = param
        result['Wine Type'] = wtype
        cpk_results.append(result)

cpk_df = pd.DataFrame(cpk_results)[[
    'Wine Type', 'Parameter', 'Mean', 'Sigma', 
    'Skewness', 'Normal Enough', 'CPU', 'CPL', 'Cpk', 'Status'
]]

# Print full results table
print("Cpk Analysis — OIV/EU Standards | Average Moving Range Method")
print("=" * 75)
print(cpk_df.to_string(index=False))




Cpk Analysis — OIV/EU Standards | Average Moving Range Method
Wine Type           Parameter   Mean  Sigma  Skewness Normal Enough   CPU   CPL   Cpk      Status
      Red    volatile acidity  0.528  0.149     0.672             ✅ 1.500 1.000 1.000 ⚠️ Marginal
    White    volatile acidity  0.278  0.082     1.577     ⚠️ Skewed 3.766 0.810 0.810 ⚠️ Marginal
      Red                  pH  3.311  0.125     0.194             ✅ 1.837 1.096 1.096   ✅ Capable
    White                  pH  3.188  0.127     0.458             ✅ 2.135 0.758 0.758 ⚠️ Marginal
      Red           sulphates  0.658  0.120     2.429     ⚠️ Skewed 2.336 1.133 1.133   ✅ Capable
    White           sulphates  0.490  0.093     0.977             ✅ 3.627 0.861 0.861 ⚠️ Marginal
      Red             alcohol 10.423  0.815     0.861             ✅ 1.872 0.787 0.787 ⚠️ Marginal
    White             alcohol 10.514  1.010     0.487             ✅ 1.481 0.665 0.665 ❌ Incapable
      Red free sulfur dioxide 15.875  8.383     1.251   

In [20]:
#cpk continued
#Cpk Bar Charts
fig = px.bar(cpk_df,
             x='Parameter',
             y='Cpk',
             color='Status',
             barmode='group',
             facet_col='Wine Type',
             title='Process Capability (Cpk) — OIV & EU Regulation 606/2009 Standards',
             labels={'Cpk': 'Cpk Value', 'Parameter': ''},
             color_discrete_map={
                 '✅ Capable':   'seagreen',
                 '⚠️ Marginal': 'orange',
                 '❌ Incapable':'crimson'
             },
             text='Cpk',
             hover_data=['Mean', 'Sigma', 'Skewness', 'Normal Enough'])

# Threshold lines
fig.add_hline(y=1.0, line_dash='dash', line_color='black')
fig.add_hline(y=0.67, line_dash='dot', line_color='grey')

# Annotations on the far right outside both panels
fig.add_annotation(
    text='Capable (1.0)',
    xref='paper', yref='y',
    x=0.90, y=1.03,
    showarrow=False,
    font=dict(size=6, color='black'),
    xanchor='left'
)
fig.add_annotation(
    text='Marginal (0.67)',
    xref='paper', yref='y',
    x=0.90, y=0.70,
    showarrow=False,
    font=dict(size=6, color='grey'),
    xanchor='left'
)

fig.update_traces(textposition='outside', textfont_size=9)

fig.update_layout(
    paper_bgcolor='#E8E8E8',
    plot_bgcolor='white',
    height=600,
    xaxis_tickangle=-35,
    xaxis2_tickangle=-35,
    legend_title='Process Status',
    margin=dict(r=150, b=120, t=80),
    # Clean up facet labels
    font=dict(size=11)
)

# Clean facet titles
fig.for_each_annotation(lambda a: a.update(
    text=a.text.replace('Wine Type=', ''),
    font=dict(size=13, color='black')
))

fig.show()

In [21]:
# Statistics Check — Mean, Std, and Data Range per Parameter
for param, spec in spec_limits.items():
    for wtype in ['Red', 'White']:
        subset = df[df['wine_type'] == wtype][param]
        mean = subset.mean()
        std = subset.std()
        print(f"{param} ({wtype})")
        print(f"  Mean: {mean:.3f} | Std: {std:.3f}")
        print(f"  LSL: {spec['LSL']} | USL: {spec['USL']}")
        print(f"  Data range: {subset.min():.3f} — {subset.max():.3f}")
        

        # Statistics Check — Mean, Std, and Data Range per Parameter



volatile acidity (Red)
  Mean: 0.528 | Std: 0.179
  LSL: 0.08 | USL: 1.2
  Data range: 0.120 — 1.580
volatile acidity (White)
  Mean: 0.278 | Std: 0.101
  LSL: 0.08 | USL: 1.2
  Data range: 0.080 — 1.100
pH (Red)
  Mean: 3.311 | Std: 0.154
  LSL: 2.9 | USL: 4.0
  Data range: 2.740 — 4.010
pH (White)
  Mean: 3.188 | Std: 0.151
  LSL: 2.9 | USL: 4.0
  Data range: 2.720 — 3.820
sulphates (Red)
  Mean: 0.658 | Std: 0.170
  LSL: 0.25 | USL: 1.5
  Data range: 0.330 — 2.000
sulphates (White)
  Mean: 0.490 | Std: 0.114
  LSL: 0.25 | USL: 1.5
  Data range: 0.220 — 1.080
alcohol (Red)
  Mean: 10.423 | Std: 1.066
  LSL: 8.5 | USL: 15.0
  Data range: 8.400 — 14.900
alcohol (White)
  Mean: 10.514 | Std: 1.231
  LSL: 8.5 | USL: 15.0
  Data range: 8.000 — 14.200
free sulfur dioxide (Red)
  Mean: 15.875 | Std: 10.460
  LSL: 10.0 | USL: 60.0
  Data range: 1.000 — 72.000
free sulfur dioxide (White)
  Mean: 35.308 | Std: 17.007
  LSL: 10.0 | USL: 60.0
  Data range: 2.000 — 289.000
chlorides (Red)
  Mean:

In [22]:
# Calculate OOS frequency (counts per parameter) per wine type
oos_results = []

for param, spec in spec_limits.items():
    for wtype in ['Red', 'White']:
        subset = df[df['wine_type'] == wtype][param]
        
        oos_below = (subset < spec['LSL']).sum()
        oos_above = (subset > spec['USL']).sum()
        oos_total = oos_below + oos_above
        oos_pct = round(oos_total / len(subset) * 100, 1)
        
        oos_results.append({
            'Parameter': param,
            'Wine Type': wtype,
            'OOS Below LSL': oos_below,
            'OOS Above USL': oos_above,
            'OOS Total': oos_total,
            'OOS %': oos_pct,
            'Sample Size': len(subset)
        })

oos_df = pd.DataFrame(oos_results)

# Print summary first
print("OOS Frequency Summary:")
print("=" * 65)
print(oos_df.to_string(index=False))



OOS Frequency Summary:
          Parameter Wine Type  OOS Below LSL  OOS Above USL  OOS Total  OOS %  Sample Size
   volatile acidity       Red              0              4          4    0.3         1599
   volatile acidity     White              0              0          0    0.0         4898
                 pH       Red              9              2         11    0.7         1599
                 pH     White             70              0         70    1.4         4898
          sulphates       Red              0              8          8    0.5         1599
          sulphates     White              2              0          2    0.0         4898
            alcohol       Red              2              0          2    0.1         1599
            alcohol     White              5              0          5    0.1         4898
free sulfur dioxide       Red            526              4        530   33.1         1599
free sulfur dioxide     White            168            346        

In [23]:
#OOS BAR CHART
fig = px.bar(oos_df,
             x='Parameter',
             y='OOS %',
             color='Wine Type',
             barmode='group',
             facet_row='Wine Type',
             title='Out-of-Spec (OOS) Frequency by Parameter — OIV/EU Standards',
             labels={'OOS %': 'OOS Rate (%)', 'Parameter': ''},
             color_discrete_map={'Red': 'crimson', 'White': 'palegoldenrod'},
             text='OOS %',
             hover_data=['OOS Below LSL', 'OOS Above USL', 'OOS Total', 'Sample Size'])


# Critical threshold line at 5%
fig.add_hline(y=5.0, line_dash='dash', line_color='red',
              annotation_text='Critical threshold (5%)',
              annotation_position='top right')

# Acceptable threshold line at 1%
fig.add_hline(y=1.0, line_dash='dot', line_color='orange',
              annotation_text='Acceptable threshold (1%)',
              annotation_position='top right')

fig.update_traces(textposition='outside', textfont_size=9)



fig.update_layout(
    paper_bgcolor='#E8E8E8',
    plot_bgcolor='white',
    height=600,
    xaxis_tickangle=-30,
    xaxis2_tickangle=-30,
    showlegend=False,
    margin=dict(r=150, b=120, t=80),
)

# Formatting labels
fig.for_each_annotation(lambda a: a.update(
    text=a.text.replace('Wine Type=', ''),
    font=dict(size=13, color='black')
))

fig.show()

In [24]:
# Final QC Summary — Merge OOC, Cpk & OOS

def verdict(oos_pct, cpk):
    if oos_pct > 5 or cpk < 0.67: return '🚨 Out of Control'
    if oos_pct > 1 or cpk < 1.0:  return '⚠️ Monitor'
    return '✅ In Control'

summary_rows = []
for param, spec in spec_limits.items():
    for wtype in ['Red', 'White']:
        ooc_row = next(r for r in results if r['Parameter'] == param and r['Wine Type'] == wtype)
        cpk_row = cpk_df[(cpk_df['Parameter'] == param) & (cpk_df['Wine Type'] == wtype)].iloc[0]
        oos_row = oos_df[(oos_df['Parameter'] == param) & (oos_df['Wine Type'] == wtype)].iloc[0]

        summary_rows.append({
            'Wine Type':  wtype,
            'Parameter':  param,
            'OOC %':      ooc_row['OOC %'],
            'Cpk':        cpk_row['Cpk'],
            'Cpk Status': cpk_row['Status'],
            'OOS %':      oos_row['OOS %'],
            'Verdict':    verdict(oos_row['OOS %'], cpk_row['Cpk'])
        })

summary_df = pd.DataFrame(summary_rows)

verdict_colors = ['#f8d7da' if v == '🚨 Out of Control'
                  else '#fff3cd' if v == '⚠️ Monitor'
                  else '#d4edda' for v in summary_df['Verdict']]

fig = go.Figure(data=[go.Table(
    columnwidth=[80, 150, 80, 80, 120, 80, 120],
    header=dict(
        values=list(summary_df.columns),
        fill_color='#404040',
        font=dict(color='white', size=12),
        align='center', height=35
    ),
    cells=dict(
        values=[summary_df[col] for col in summary_df.columns],
        fill_color=[['#f9f9f9'] * len(summary_df)] * 6 + [verdict_colors],
        font=dict(size=11),
        align='center', height=30
    )
)])

fig.update_layout(
    title='Phase 3 QC Summary — Process Stability, Capability & Compliance',
    paper_bgcolor='#E8E8E8',
    height=500
)
fig.show()

print("\nPhase 3 QC Summary:")
print("=" * 75)
print(summary_df.to_string(index=False))

summary_df.to_csv('../data/processed/spc_summary.csv', index=False)
print("✅ SPC summary saved")


Phase 3 QC Summary:
Wine Type           Parameter  OOC %   Cpk  Cpk Status  OOS %          Verdict
      Red    volatile acidity    0.6 1.000 ⚠️ Marginal    0.3     ✅ In Control
    White    volatile acidity    1.7 0.810 ⚠️ Marginal    0.0       ⚠️ Monitor
      Red                  pH    0.5 1.096   ✅ Capable    0.7     ✅ In Control
    White                  pH    0.7 0.758 ⚠️ Marginal    1.4       ⚠️ Monitor
      Red           sulphates    1.7 1.133   ✅ Capable    0.5     ✅ In Control
    White           sulphates    1.0 0.861 ⚠️ Marginal    0.0       ⚠️ Monitor
      Red             alcohol    0.5 0.787 ⚠️ Marginal    0.1       ⚠️ Monitor
    White             alcohol    0.0 0.665 ❌ Incapable    0.1 🚨 Out of Control
      Red free sulfur dioxide    1.4 0.234 ❌ Incapable   33.1 🚨 Out of Control
    White free sulfur dioxide    0.7 0.591 ❌ Incapable   10.5 🚨 Out of Control
      Red           chlorides    1.9 1.108   ✅ Capable    2.6       ⚠️ Monitor
    White           chlorides  